# 第 31 章 · 读 Inductor 生成的代码：它已经帮你融了什么

本章回答：`torch.compile` 生成的代码在哪里看？一份生成代码长什么样，怎么读出"哪些算子被融成了一个 kernel、哪些交给了现成库"？什么时候 compile 已经够好、不必手写？

配套讲义：仓库里 `pytorch-deep-dive/notes/lessons/31-read-inductor.md`（讲解、动画、练习都在讲义里）。对应原文：AISPE ch14 §14.1.4「TorchInductor 后端代码生成」（`ai-infra/materials/ai-systems-performance-engineering/docs/chapters/ch14.md` 第 298–441 行，自动调优第 365 行）、§14.1.8「性能提示和调试生成的代码」（第 552–573 行）、「调试编译器阶段」（第 1029–1108 行）；本章生成代码片段是本机 torch 2.15.0（CPU 与 MPS）实测

**怎么用**：在 `pytorch-deep-dive/solutions/chapter-code/` 下用 `uv run jupyter lab` 打开（或任何装了 torch 的环境）。从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。Mac 上能跑：计时类的格子在 Mac 上用 MPS，分布式的格子在本机起 2 个 Gloo 进程。

In [ ]:
import re

from torch._inductor.utils import run_and_get_code
import torch
import torch.nn.functional as F

torch.set_printoptions(precision=4, sci_mode=False)
print("准备好了", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 公式 / 说法 | 写法 | 跑出来 |
|---|---|---|---|
| 第二步 softmax | 整条链一个 kernel | `summarize(generated(softmax2, x))` | `cpp_fused__softmax_abs_all_amax_eq_mul_ne_sub_0`，向量化 |
| 第二步 gelu(x@w+b) | 矩阵乘调库 | `summarize(generated(mlp, ...))` | extern `addmm` + `cpp_fused_gelu_0` |

## 1. 先把要用的东西定义好

这几格只是定义（函数、类、常量），运行后没有输出也正常。用到的前几章的函数也一起抄进来了，不需要本地文件。

**函数 `generated`**（来自 `ch31_read_inductor.py`）：下面要用到的小工具。

In [ ]:
def generated(fn, *args) -> str:
    torch._dynamo.reset()
    _, code = run_and_get_code(torch.compile(fn), *args)
    return code[0]

**函数 `mlp`**（来自 `ch31_read_inductor.py`）：下面要用到的小工具。

In [ ]:
def mlp(x, w, b):
    return F.gelu(x @ w + b)

**函数 `softmax2`**（来自 `ch31_read_inductor.py`）：下面要用到的小工具。

In [ ]:
def softmax2(x):
    return torch.softmax(x * 2, -1)

**函数 `summarize`**（来自 `ch31_read_inductor.py`）：下面要用到的小工具。

In [ ]:
def summarize(src: str) -> dict:
    return dict(fused=sorted(set(re.findall(r"cpp_fused\w*", src))),
                extern=sorted(set(re.findall(r"extern_kernels\.(\w+)", src))),
                vectorized="at::vec::Vectorized" in src)

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第二步 softmax(x * 2)：", summarize(generated(softmax2, torch.randn(8, 64))))
print("gelu(x @ w + b)：", summarize(generated(mlp, torch.randn(8, 64), torch.randn(64, 32), torch.randn(32))))

## 3. 代码里才看得到的

1. **同一个函数在 MPS 上生成 Metal、在 CUDA 上生成 Triton**：名字前缀不同，规则相同。

## 4. 自己改一改

- `generated(lambda x: torch.relu(x) * 2 + 1, torch.randn(64))`，看 kernel 名字。
- 把 `mlp` 的 gelu 去掉，看还剩几个 kernel。

讲义末尾的练习题可以在下面的空格子里用上面的函数验算——先自己在纸上算，再来核对。

In [ ]:
# 在这里写你的代码

## 口诀

> 拿代码：run_and_get_code、output_code、COMPILE_DEBUG；
> 名字里列着融进去的算子，矩阵乘走 extern 调库；
> 数 kernel、数 buf、找 clone；改算法级的融合，compile 不会，用库或手写。